# Anomalías industriales en sensores: Modelado y selección

Ejecuta la etapa anterior primero. Las cifras y conclusiones siguientes se calculan con los datos de esta ejecución; no edites manualmente sus valores. Consulta [metodología](../docs/metodologia.md) para interpretar resultados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("industrial_anomaly_detection")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from industrial_anomaly_detection.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from industrial_anomaly_detection.analysis import run_stage
import pandas as pd
metrics, comment = run_stage(PROJECT_ROOT, "modelado")
display(Markdown(comment))
scalars = {k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}
if scalars:
    display(pd.DataFrame(list(scalars.items()), columns=["Indicador", "Valor"]))
if "candidatos" in metrics:
    display(pd.DataFrame(metrics["candidatos"]))
if "baseline" in metrics and "modelo" in metrics:
    comparison = {key: {k: v for k, v in metrics[key].items() if not isinstance(v, (dict, list))} for key in ["baseline", "modelo"]}
    display(pd.DataFrame(comparison).T)
if "clasificacion" in metrics.get("modelo", {}):
    display(pd.DataFrame(metrics["modelo"]["clasificacion"]).T)
for image_name in ["eda.png", "evaluacion.png"]:
    image_path = PROJECT_ROOT / "reports/modelado" / image_name
    if image_path.exists():
        display(Image(filename=str(image_path)))


# Anomalías industriales en sensores: modelado

Isolation Forest se ajusta solo con filas normales. Se elige el umbral **0.6240** por F1 de validación; test no interviene. Esta cadena es **semisupervisada**: utiliza etiquetas para definir normalidad y seleccionar umbrales, aunque el detector no aprende una clasificación supervisada de fallos.


,Indicador,Valor
0,criterio,f1_validacion
1,threshold_elegido,0.623986
2,quantile_elegido,0.99
3,normales_entrenamiento,5797


,quantile,threshold,casos,precision,recall,f1,average_precision,matriz_confusion,alertas,fallos_reales
0,0.950,0.569647,2000,0.144068,0.250000,0.182796,0.172903,"[[1831, 101], [51, 17]]",118,68
1,0.975,0.593525,2000,0.180328,0.161765,0.170543,0.172903,"[[1882, 50], [57, 11]]",61,68
2,0.990,0.623986,2000,0.333333,0.132353,0.189474,0.172903,"[[1914, 18], [59, 9]]",27,68


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/04_modelado.json")
from industrial_anomaly_detection.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))


[Abrir informe integral](../reports/informe_final/informe_final.md)